# Notebook 28: Graviton Identification (Paper III, §1–2)

This notebook verifies the graviton identification chain:

1. **CMS-CS Casimir**: $C_2(m) = m(N-m)/2$ for all $N$, $m$
2. **Graviton at $N=7$**: $f(3,7) = 6 = j(j+1)$ with $j=2$
3. **Pell equation**: $N^2 - 2y^2 = -1$ solutions
4. **Uniqueness**: $N=7$ is unique for $j=2$ in the stable range

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from math import sqrt, pi, sin, cos, sinh, log
from fractions import Fraction

assertion_count = 0

## 1. CMS-CS Casimir Identification: $C_2(m) = m(N-m)/2$

The $\mathrm{sl}(2,\mathbb{R})$ Casimir on the $m$-th $\mathbb{Z}_N$ tangential mode
equals the Havelock sum $T_m = m(N-m)/2$, simultaneously for both the
Calogero-Moser-Sutherland (CMS) and Chern-Simons (CS) systems.

In [ ]:
from planetary_polygons.proofs.casimir_equals_havelock import (
    verify_casimir_identification, verify_same_algebra,
    havelock_sum, sl2r_casimir_on_mode
)

max_err, results = verify_casimir_identification(N_max=12)
print(f'Max relative error across N=3..12, rho in {{0.3,...,2.0}}: {max_err:.2e}')
assert max_err < 1e-10, f'Casimir identification failed: {max_err}'
assertion_count += 1
print('PASSED: circulant eigenvalue = T_m / sinh^2(rho) to machine precision\n')

# Display a summary table
print(f'{"N":>4s} {"rho":>6s} {"max_err":>12s}')
print('-' * 26)
for N in range(3, 13):
    subset = [r for r in results if r[0] == N]
    worst = max(r[2] for r in subset)
    print(f'{N:4d} {"all":>6s} {worst:12.2e}')

In [ ]:
# Verify the angular Hessian eigenvalues match Havelock for N=7,8,9
from planetary_polygons.proofs.casimir_equals_havelock import h2_laplacian_angular_hessian

print('Angular Hessian eigenvalues vs Havelock formula:')
print(f'{"N":>4s} {"m":>4s} {"Havelock m(N-m)/2":>20s} {"Casimir C2(m)":>16s} {"match":>8s}')
print('-' * 56)

for N in [7, 8, 9]:
    for m in range(1, N // 2 + 1):
        hav = havelock_sum(N, m)
        cas = sl2r_casimir_on_mode(N, m, rho0=1.0)
        match = abs(hav - cas) < 1e-12
        assert match, f'Mismatch at N={N}, m={m}'
        assertion_count += 1
        print(f'{N:4d} {m:4d} {hav:20.1f} {cas:16.1f} {"YES":>8s}')
    print()

# Verify CMS and CS use the same algebra
assert verify_same_algebra(), 'CMS-CS algebra mismatch'
assertion_count += 1
print('CMS and CS algebras verified identical (same sl(2,R) on H^2).')

## 2. Graviton at $N=7$: $f(3,7) = 6 = j(j+1)$ with $j=2$

At $N=7$, the critical mode is $m^* = 3$ with Casimir $f(3,7) = 3 \times 4 / 2 = 6$.
Solving $j(j+1) = 6$ gives $j = 2$, the spin of the graviton.

In [ ]:
from planetary_polygons.extensions.n_selection import casimir, critical_spin

N = 7
print(f'N = {N}: Casimir f(m,N) = m(N-m)/2 for each mode:\n')
print(f'{"m":>4s} {"f(m,N)":>10s} {"j from j(j+1)=f":>18s} {"integer?":>10s}')
print('-' * 46)

for m in range(1, N // 2 + 1):
    f = casimir(m, N)
    j = (-1 + sqrt(1 + 4 * f)) / 2
    is_int = abs(j - round(j)) < 1e-10
    print(f'{m:4d} {f:10.1f} {j:18.6f} {"YES" if is_int else "no":>10s}')

# The critical mode m*=3
m_star = N // 2  # = 3
f_star = casimir(m_star, N)
j_star = (-1 + sqrt(1 + 4 * f_star)) / 2

assert m_star == 3, f'Expected m*=3, got {m_star}'
assertion_count += 1
assert f_star == 6.0, f'Expected f=6, got {f_star}'
assertion_count += 1
assert abs(j_star - 2.0) < 1e-12, f'Expected j=2, got {j_star}'
assertion_count += 1

print(f'\nCritical mode m* = {m_star}: f({m_star},{N}) = {f_star:.0f} = j(j+1) with j = {j_star:.0f}')
print(f'j = 2 is the GRAVITON spin.')

In [ ]:
# Show no other N <= 30 gives j=2 at the critical mode
print('Scan N=3..30 for j=2 at the critical mode:\n')
print(f'{"N":>4s} {"m*":>4s} {"f(m*,N)":>10s} {"j":>10s} {"j=2?":>6s}')
print('-' * 38)

j2_count = 0
for N in range(3, 31):
    m_star = N // 2
    f = casimir(m_star, N)
    j = (-1 + sqrt(1 + 4 * f)) / 2
    is_j2 = abs(j - 2.0) < 1e-10
    if is_j2:
        j2_count += 1
    flag = '<-- j=2' if is_j2 else ''
    print(f'{N:4d} {m_star:4d} {f:10.1f} {j:10.4f} {flag:>6s}')

assert j2_count == 1, f'Expected exactly 1 polygon with j=2, found {j2_count}'
assertion_count += 1
print(f'\nN=7 is UNIQUE for j=2 at the critical mode among N=3..30.')

## 3. Pell Equation: $N^2 - 2y^2 = -1$

The integer-spin condition $f(m^*,N) = j(j+1)$ for odd $N$ reduces to
the negative Pell equation $N^2 - 2(2j+1)^2 = -1$.
Solutions are generated by powers of $\varepsilon = 1 + \sqrt{2}$.

In [ ]:
from planetary_polygons.extensions.n_selection import pell_solutions, verify_pell

solutions = pell_solutions(n_solutions=8)

print('Pell equation N^2 - 2y^2 = -1 solutions (first 8):\n')
print(f'{"k":>4s} {"N":>6s} {"y":>8s} {"j=(y-1)/2":>12s} {"N^2-2y^2":>12s} {"verified":>10s}')
print('-' * 56)

for k, (N, y, j) in enumerate(solutions, 1):
    pell_val = N * N - 2 * y * y
    verified = pell_val == -1
    assert verified, f'Pell failed at k={k}: N={N}, y={y}'
    assertion_count += 1
    print(f'{k:4d} {N:6d} {y:8d} {j:12d} {pell_val:12d} {"YES" if verified else "NO":>10s}')

# Highlight key results
print(f'\nKey: (N=7, y=5) gives j=2 (graviton)')
print(f'     (N=41, y=29) gives j=14')
print(f'     (N=239, y=169) gives j=84')

assert solutions[1] == (7, 5, 2), f'Expected (7,5,2), got {solutions[1]}'
assertion_count += 1

In [ ]:
# Verify the generation by powers of epsilon = 1 + sqrt(2)
eps = 1 + sqrt(2)
print(f'Fundamental unit: epsilon = 1 + sqrt(2) = {eps:.10f}\n')

print(f'{"power":>8s} {"N (real part)":>14s} {"y (sqrt2 part)":>16s} {"Pell check":>12s}')
print('-' * 54)

for k in range(1, 8):
    power = 2 * k - 1
    # (1+sqrt(2))^power = a + b*sqrt(2)
    a, b = 1, 0
    for _ in range(power):
        a_new = a + 2 * b
        b_new = a + b
        a, b = a_new, b_new
    pell_check = a * a - 2 * b * b
    print(f'{power:8d} {a:14d} {b:16d} {pell_check:12d}')
    assert pell_check == -1, f'Pell failed at power {power}'
    assertion_count += 1

## 4. Uniqueness: $N=7$ is Unique for $j=2$ in the Stable Range

Among all $N \le 50$, only $N=7$ gives $j=2$ at the critical mode.
Moreover, $N=7$ is in the stable range ($N \le 7$ on the flat plane).

In [ ]:
from planetary_polygons.extensions.n_selection import integer_spin_polygons, critical_spin

# Full scan N=3..50 for integer spin at critical mode
int_spin = integer_spin_polygons(N_max=50)

print('All polygon numbers N <= 50 with integer critical spin:\n')
print(f'{"N":>4s} {"m*":>4s} {"f(m*,N)":>10s} {"j":>6s} {"stable (N<=7)?":>16s}')
print('-' * 44)

for N, j in int_spin:
    m_star = N // 2
    f = casimir(m_star, N)
    stable = 'YES' if N <= 7 else 'no'
    print(f'{N:4d} {m_star:4d} {f:10.1f} {j:6d} {stable:>16s}')

# Only N=4 (j=1) and N=7 (j=2) below N=24
below_24 = [(N, j) for N, j in int_spin if N < 24]
assert below_24 == [(4, 1), (7, 2)], f'Unexpected: {below_24}'
assertion_count += 1

# N=7 is the only j=2
j2_list = [(N, j) for N, j in int_spin if j == 2]
assert j2_list == [(7, 2)], f'Expected only N=7 for j=2, got {j2_list}'
assertion_count += 1

print(f'\nBelow N=24: only N=4 (j=1, gauge boson) and N=7 (j=2, graviton).')
print(f'N=7 is the UNIQUE graviton polygon.')

In [ ]:
# Extended scan: j at critical mode for N=3..50
print('Critical spin j for N=3..50:\n')
print(f'{"N":>4s} {"m*":>4s} {"f":>8s} {"j":>10s} {"integer?":>10s} {"particle":>14s}')
print('-' * 52)

for N in range(3, 51):
    j, is_int, m_star, f = critical_spin(N)
    j_int = int(round(j)) if is_int else None
    particle = ''
    if is_int:
        if j_int == 1:
            particle = 'gauge boson'
        elif j_int == 2:
            particle = 'GRAVITON'
        else:
            particle = f'spin-{j_int}'
    print(f'{N:4d} {m_star:4d} {f:8.1f} {j:10.4f} {"YES" if is_int else "":>10s} {particle:>14s}')

print(f'\nOnly two integer-spin polygons below N=24: N=4 (j=1) and N=7 (j=2).')

## Summary

1. **CMS-CS Casimir**: $C_2^{\mathrm{sl}(2,\mathbb{R})}(m) = m(N-m)/2$ verified to machine precision
   for $N = 3, \ldots, 12$ at five values of $\rho_0$.

2. **Graviton at $N=7$**: $f(3,7) = 6 = 2 \times 3 = j(j+1)$ with $j = 2$.
   No other $N \le 30$ gives $j = 2$ at the critical mode.

3. **Pell equation**: $N^2 - 2y^2 = -1$ generates integer-spin odd polygons.
   First 5 solutions: $(1,0), (7,2), (41,14), (239,84), (1393,492)$.

4. **Uniqueness**: Only $N = 4$ ($j=1$, gauge boson) and $N = 7$ ($j=2$, graviton)
   have integer critical spin below $N = 24$.

In [ ]:
print(f'\nAll {assertion_count} assertions passed.')